In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(-1, '/Users/ponddie/Documents/py/mda_project/explo/yenha/lib')
from hahelper import *

import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_columns=100
# pd.options.display.max_rows=100

In [2]:
df_raw = pd.read_csv(PATH_TRAFFIC + "data-2026-01.csv", header=None, names=TRAFFIC_COLS)
df_sites = pd.read_csv(FILEPATH_SITES, header=None, names=SITES_COLS)
df_rich = pd.read_csv(FILEPATH_RICH, header=None, names=RICH_COLS)

## 1. Label

In [7]:
allmonth = [(f"{y}-{m:02}") for y in range(2019, 2027) for m in range(1, 13) if f"{y}-{m:02}" >= "2019-08" and f"{y}-{m:02}" <= "2026-03"]
print(len(allmonth)), print(allmonth)

df_all = []
for i in allmonth:
    df = pd.read_csv(PATH_TRAFFIC + f"data-{i}.csv", header=None, names=TRAFFIC_COLS)
    df = df.query('type=="FIETSERS"')
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["datehour"] = (pd.to_datetime(df["start_time"].dt.strftime("%Y-%m-%d")) + pd.to_timedelta(df["start_time"].dt.hour, unit="h"))

    df["year"] = df["start_time"].dt.year
    df["month"] = df["start_time"].dt.strftime("%m")
    df["date"] = df["start_time"].dt.strftime("%d")
    df["hour"] = df["start_time"].dt.hour

    df_hourly = (
        df.groupby(["site_id",  "datehour", "year", "month", "date", "hour", "direction"])["traffic"].sum().unstack(fill_value=0).reset_index()
        .rename(columns={"IN": "traffic_in", "OUT": "traffic_out"})
    )
    df_hourly["traffic_total"] = df_hourly["traffic_in"] + df_hourly["traffic_out"]
    df_hourly = df_hourly.sort_values(["site_id",  "datehour", "year", "month", "date", "hour"]).reset_index(drop=True)
    print(f'Done {i}')
    df_all.append(df_hourly)

df_merged = pd.concat(df_all, ignore_index=True)
df_merged.to_csv('/Users/ponddie/Documents/py/mda_project/explo/yenha/feature/label_hourly_201908_202603.csv', index=False)
print(df_merged.shape)
df_merged.head()

Done month 2022-06
Done month 2022-07
Done month 2022-08
Done month 2022-09
Done month 2022-10
Done month 2022-11
Done month 2022-12
Done month 2023-01
Done month 2023-02
Done month 2023-03
Done month 2023-04
Done month 2023-05
Done month 2023-06
Done month 2023-07
Done month 2023-08
Done month 2023-09
Done month 2023-10
Done month 2023-11
Done month 2023-12
Done month 2024-01
Done month 2024-02
Done month 2024-03
Done month 2024-04
Done month 2024-05
Done month 2024-06
Done month 2024-07
Done month 2024-08
Done month 2024-09
Done month 2024-10
Done month 2024-11
Done month 2024-12
Done month 2025-01
Done month 2025-02
Done month 2025-03
Done month 2025-04
Done month 2025-05
Done month 2025-06
Done month 2025-07
Done month 2025-08
Done month 2025-09
Done month 2025-10
Done month 2025-11
Done month 2025-12
Done month 2026-01
Done month 2026-02
(4447687, 9)


direction,site_id,datehour,year,month,date,hour,traffic_in,traffic_out,traffic_total
0,1,2022-06-01 00:00:00,2022,06,01,0,0.0,2.0,2.0
1,1,2022-06-01 01:00:00,2022,06,01,1,0.0,0.0,0.0
2,1,2022-06-01 02:00:00,2022,06,01,2,0.0,0.0,0.0
3,1,2022-06-01 03:00:00,2022,06,01,3,0.0,0.0,0.0
4,1,2022-06-01 04:00:00,2022,06,01,4,0.0,1.0,1.0


In [105]:
df_merged.groupby('month').agg(site_id_nunique=('site_id', 'nunique'), site_id_size=('site_id', 'size'))

,site_id_nunique,site_id_size
month,,
2022-06,100,60866
2022-07,109,79020
2022-08,115,83624
2022-09,129,89268
2022-10,129,95976
2022-11,129,92545
2022-12,129,95390
2023-01,134,97551
2023-02,134,90048


- Use data from 2022-06	to 2026-02

## 2. Run feature

In [4]:
lmonth = [(f"{y}-{m:02}") for y in range(2022, 2027) for m in range(1, 13) if f"{y}-{m:02}" >= "2022-06" and f"{y}-{m:02}" < "2026-03"]
len(lmonth), print(lmonth)

['2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02']


(45, None)

### 2.1 compute_traffic_15min_features

In [5]:
dfs = []
for i in lmonth:
    _df = compute_traffic_15min_features(pd.read_csv(PATH_TRAFFIC + f"data-{i}.csv", header=None, names=TRAFFIC_COLS))
    dfs.append(_df)
    # print(f'Done {i}')
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv('/Users/ponddie/Documents/py/mda_project/explo/yenha/feature/fts_traffic_agg15min.csv', index=False)
print(df_all.shape) 
df_all.head()

(4447687, 20)


,site_id,datehour,traffic_in_00_15,traffic_out_00_15,traffic_in_15_30,traffic_out_15_30,traffic_in_30_45,traffic_out_30_45,traffic_in_45_60,traffic_out_45_60,traffic_in,traffic_out,traffic_in_15m_mean,traffic_in_15m_std,traffic_in_15m_min,traffic_in_15m_max,traffic_out_15m_mean,traffic_out_15m_std,traffic_out_15m_min,traffic_out_15m_max
0,1,2022-06-01 00:00:00,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.50,0.57735,0.0,1.0
1,1,2022-06-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0
2,1,2022-06-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0
3,1,2022-06-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0
4,1,2022-06-01 04:00:00,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.25,0.50000,0.0,1.0


### 2.2 compute_datetime_features

In [6]:
dfs = []
for i in lmonth:
    _df = compute_datetime_features(pd.read_csv(PATH_TRAFFIC + f"data-{i}.csv", header=None, names=TRAFFIC_COLS))
    dfs.append(_df)
    # print(f'Done {i}')
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv('/Users/ponddie/Documents/py/mda_project/explo/yenha/feature/fts_datetime.csv', index=False)
print(df_all.shape) 
df_all.head()

(4447687, 14)


,site_id,datehour,dt_is_weekend,dt_is_weekday,dt_is_monday,dt_is_friday,dt_is_morning_rush,dt_is_afternoon_rush,dt_is_rush_hour,dt_is_lunch_hour,dt_is_night,dt_is_business_hours,dt_is_public_holiday,dt_public_holiday
0,1,2022-06-01 00:00:00,0,1,0,0,0,0,0,0,1,0,0,None
1,1,2022-06-01 01:00:00,0,1,0,0,0,0,0,0,1,0,0,None
2,1,2022-06-01 02:00:00,0,1,0,0,0,0,0,0,1,0,0,None
3,1,2022-06-01 03:00:00,0,1,0,0,0,0,0,0,1,0,0,None
4,1,2022-06-01 04:00:00,0,1,0,0,0,0,0,0,1,0,0,None


In [110]:
df_all.query('datehour=="2026-01-01 00:00:00"').shape

(145, 14)

### 2.3 compute_site_features

In [ ]:
dfs = []
for i in lmonth:
    _df = compute_site_features(pd.read_csv(PATH_TRAFFIC + f"data-{i}.csv", header=None, names=TRAFFIC_COLS))
    dfs.append(_df)
    # print(f'Done {i}')
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv('/Users/ponddie/Documents/py/mda_project/explo/yenha/feature/fts_site.csv', index=False)
print(df_all.shape) 
df_all.head()

Done 2022-06
Done 2022-07
Done 2022-08
Done 2022-09
Done 2022-10
Done 2022-11
Done 2022-12
Done 2023-01
Done 2023-02
Done 2023-03
Done 2023-04
Done 2023-05
Done 2023-06
Done 2023-07
Done 2023-08
Done 2023-09
Done 2023-10
Done 2023-11
Done 2023-12
Done 2024-01
Done 2024-02
Done 2024-03
Done 2024-04
Done 2024-05
Done 2024-06
Done 2024-07
Done 2024-08
Done 2024-09
Done 2024-10
Done 2024-11
Done 2024-12
Done 2025-01
Done 2025-02
Done 2025-03
Done 2025-04
Done 2025-05
Done 2025-06
Done 2025-07
Done 2025-08
Done 2025-09
Done 2025-10
Done 2025-11
Done 2025-12
Done 2026-01
Done 2026-02
(4447687, 7)


,site_id,datehour,site_sensor_age,site_is_road_tunnel,site_is_road_national,site_is_road_ring,site_is_road_motorway
0,1,2022-06-01 00:00:00,1014,1,0,0,0
1,1,2022-06-01 01:00:00,1014,1,0,0,0
2,1,2022-06-01 02:00:00,1014,1,0,0,0
3,1,2022-06-01 03:00:00,1014,1,0,0,0
4,1,2022-06-01 04:00:00,1014,1,0,0,0
